# Quick recommendations for seismology data management

This short notebook gives the main recommendations for organizing continuous and event waveform data in this course.

## 1. Big picture

A useful pattern is to treat **continuous waveform data** and **event waveform data** as two related but distinct products.

### Continuous waveform data
Continuous data are best treated as a long-term archive. They should be stored in a standardized layout that is easy to read by many tools.

### Event waveform data
Event data are best treated as extracted products associated with individual detections or cataloged events. These are often easier to review, pick, classify, and relocate when grouped event-by-event.

This leads naturally to using:

- **SDS archives** for continuous data, and
- **SEISAN-style event files** for event waveform data.

## 2. Recommendation for continuous data

### Store continuous data as:

- **one MiniSEED file per day per SEED id**
- organized in **SeisComP Data Structure (SDS)** layout

This is a very good default for long-term continuous archives because it is:

- simple,
- standard,
- compatible with ObsPy and many other tools,
- efficient for day-based processing,
- well suited to RSAM, detection, and archive conversion workflows.

A typical SDS path looks like:

```text
SDS_ROOT/YYYY/NET/STA/CHAN.D/NET.STA.LOC.CHAN.D.YYYY.JJJ
```

where:

- `YYYY` = 4-digit year
- `NET` = network code
- `STA` = station code
- `LOC` = location code
- `CHAN` = channel code
- `JJJ` = Julian day

### Why this is recommended

A one-file-per-day-per-SEED-id structure keeps the archive modular and easy to repair. If one day is corrupt or missing, you only lose one day for one channel, not an entire month or station.

It also matches the natural rhythm of many continuous-data workflows:

- reading one day at a time,
- computing daily metrics,
- running detectors day-by-day,
- parallelizing work across days or channels.

## 3. SDS and Antelope can work together

An SDS archive does **not** prevent you from using Antelope.

A useful strategy is:

1. archive the continuous data in **MiniSEED in SDS layout**,
2. then build an **Antelope `wfdisc` table** that points to those MiniSEED files.

This gives you the best of both worlds:

- **SDS** as a clean archival structure,
- **Antelope** as a database/query layer.

So SDS and Antelope are not competing ideas. They can work very well together.

### Practical consequence

You do **not** need to duplicate the waveform data just to use Antelope. In many cases, it is enough to maintain:

- one authoritative SDS archive, and
- a `wfdisc` table referring to those files.

## Event data: the short recommendation

Store **event waveform data** as **one MiniSEED file per event**, not one file per channel or per station. Use a **SEISAN-style event filename** and store the files in:

```text
WAV/{DBNAME}/{YYYY}/{MM}
```

where:

- `DBNAME` is the database name, up to 5 characters
- `YYYY` is the 4-digit year
- `MM` is the 2-digit month

Store the parallel event metadata in:

```text
REA/{DBNAME}/{YYYY}/{MM}
```

The `REA` tree contains S-files in Nordic format, while the `WAV` tree contains the waveform data for those events. This event-based organization works especially well with SEISAN and remains very practical for manual review, picking, and catalog analysis.


## 8. Practical recommendations for students

### Recommendation 1
Keep one **authoritative archive** for continuous data.

That should usually be your SDS MiniSEED archive.

### Recommendation 2
Treat extracted event waveform files as **derived products**.

These can be regenerated from the continuous archive if needed, provided the event timing and extraction rules are documented.

### Recommendation 3
Do not invent a custom folder structure unless there is a very strong reason.

Use established conventions where possible:

- SDS for continuous data
- SEISAN `WAV/REA` for event data and metadata

### Recommendation 4
Use naming conventions consistently.

Small inconsistencies in network, station, location, or channel naming become major problems later.

### Recommendation 5
Keep metadata as close to the waveform data as possible.

At minimum, preserve:

- station metadata,
- instrument responses,
- event IDs,
- extraction times,
- provenance of picks and locations.

### Recommendation 6
Prefer open and well-supported formats.

- MiniSEED for waveforms
- StationXML for station metadata
- Nordic or QuakeML for event metadata, depending on workflow

## 11. Final recommendations

If you remember only three things from this notebook, remember these:

### Continuous data
Archive them as **daily MiniSEED files **per SEED id** in SDS format**.

### Event data
Store them as **one MiniSEED file per event** in a **SEISAN-compatible `WAV` directory tree**.

### Event metadata
Store readings/results in **parallel `REA` directories** so that event interpretation stays tied to event waveform files.

This combination gives you a practical and durable workflow that works well with:

- Python/ObsPy,
- Antelope,
- and SEISAN.